In [ ]:
from collections import deque
import sys
sys.setrecursionlimit(10**7)


class FordFulkerson: # i used gpt to help with this class
    def __init__(self, num_nodes):
        self.num_nodes = num_nodes
        self.capacity = [[0] * num_nodes for i in range(num_nodes)]
        self.adjacency_list = [[] for i in range(num_nodes)]

    def add_edge(self, u, v, cap):
        self.capacity[u][v] += cap  # accommodate multiple edges
        self.adjacency_list[u].append(v)
        self.adjacency_list[v].append(u)  # residual backward edge

    def dfs_find_path(self, node, sink, flow, visited):
        if node == sink:
            return flow

        visited[node] = True

        for neighbor in self.adjacency_list[node]:
            residual = self.capacity[node][neighbor]
            if residual > 0 and not visited[neighbor]:
                pushed = self.dfs_find_path(neighbor, sink, min(flow, residual), visited)
                if pushed > 0:
                    # Update residual graph
                    self.capacity[node][neighbor] -= pushed
                    self.capacity[neighbor][node] += pushed
                    return pushed

        return 0

    def max_flow(self, source, sink):
        total_flow = 0
        INF = 10**18

        while True:
            visited = [False] * self.num_nodes
            pushed = self.dfs_find_path(source, sink, INF, visited)
            if pushed == 0:
                break
            total_flow += pushed

        return total_flow

    def source_reachable_in_residual(self, source):
        # Nodes reachable from source in residual graph -> source side of min-cut.
        visited = [False] * self.num_nodes
        queue = deque([source])
        visited[source] = True

        while queue:
            node = queue.popleft()
            for neighbor in self.adjacency_list[node]:
                if not visited[neighbor] and self.capacity[node][neighbor] > 0:
                    visited[neighbor] = True
                    queue.append(neighbor)

        return visited


def solve_from_string(input_str):
    inp = input_str.strip().split()
    t = iter(inp)

    num_vertices = int(next(t))
    penalty_cost = int(next(t))

    profit = [0] * num_vertices
    for i in range(num_vertices):
        v = int(next(t))
        p = int(next(t))
        profit[v] = p

    # Read all (u, v) edges
    directed_edges = []
    while True:
        try:
            u = int(next(t))
            v = int(next(t))
            directed_edges.append((u, v))
        except StopIteration:
            break

    # Build adjacency list for DAG
    adjacency = [[] for i in range(num_vertices)]
    for u, v in directed_edges:
        adjacency[u].append(v)

    # Compute transitive closure by DFS from each vertex
    reachable = [set() for i in range(num_vertices)]

    for start in range(num_vertices):
        stack = [start]
        seen = [False] * num_vertices
        seen[start] = True

        while stack:
            node = stack.pop()
            for nxt in adjacency[node]:
                if not seen[nxt]:
                    seen[nxt] = True
                    reachable[start].add(nxt)
                    stack.append(nxt)

    # Build flow network
    source = num_vertices
    sink = num_vertices + 1
    ff = FordFulkerson(num_nodes=num_vertices + 2)

    total_positive_profit = 0

    for v in range(num_vertices):
        if profit[v] >= 0:
            ff.add_edge(source, v, profit[v])
            total_positive_profit += profit[v]
        else:
            ff.add_edge(v, sink, -profit[v])

    # Add penalty edges L for each reachable dependency
    for u in range(num_vertices):
        for v in reachable[u]:
            ff.add_edge(u, v, penalty_cost)

    # Compute max flow => min cut
    mincut_cost = ff.max_flow(source, sink)
    max_profit = total_positive_profit - mincut_cost

    # Extract selected vertices (source side of min cut)
    reachable_from_source = ff.source_reachable_in_residual(source)
    selected_vertices = [v for v in range(num_vertices) if reachable_from_source[v]]

    return max_profit, selected_vertices


In [ ]:
# import os
# print(os.getcwd())
import sys

path = input("input relative file path: ")
with open(path, 'r') as f:
    content = f.read()


print(solve_from_string(content))

(233729, [0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 103, 113, 120])
